# 04 finetune v2 — CLIP-L/14 @ 336 + RoBERTa-large + asym depth-4 + COCO pretrain init

**Purpose.** VQA finetuning on top of the pretrained encoders + fusion + pooling produced by
`04_pretrain_coco_itc_itm.ipynb`. Same answer vocab (3129), same `normalize_answer`, same eval
metric and splits as `03_train_clip_upgrade.ipynb` for direct A/B comparison.

**What changed vs `03_*`.**

1. Vision: ViT-L/14 @ 336 (was ViT-B/16 @ 384). Token count preserved at 577 (24×24+1).
2. Text: RoBERTa-large (was base). Hidden dim read dynamically (1024).
3. Fusion: depth-4 asymmetric stack (was 1). `EMBED_DIM = 768`, `NUM_HEADS = 12`.
4. Pooling: learnable attention pool on each stream (was CLS-of-each concatenation).
5. Initialization: from `pretrain_final.pt` (COCO captions ITC + ITM), unless the pretrain gate
   failed — in which case we fall back to LAION-2B init and the run name flips to `pt0`.
6. AMP: `bfloat16` autocast on L4 (no `GradScaler`, no loss-scaling unscale step).
7. LLRD on both encoders (decay 0.95) — top layers train at full rate, bottom layers nearly frozen.
8. Aug: `RandomHorizontalFlip` dropped (contradicts the `RandAugmentNoGeom` rationale —
   horizontal flip is the most destructive op for "what's on the left/right" questions).
9. Loaders: `NUM_WORKERS = 2`, `persistent_workers=True`, `prefetch_factor=4`, single shared tokenizer.
10. Auto-resume: regex restricted to the new run name + tightened to `_epoch\d+$` (defensive).

**Realistic accuracy target.** 71–74% val VQA acc with pretrained init; 70–72% if the Phase 2 gates
failed and we fell back to LAION-2B init. **Not 80%** — 80% would require a pretrained VLM (BLIP,
ALBEF, BLIP-2) as the starting point, which is a different project and out of scope here.

**Phase 1 fallbacks taken (fill in after profiling):** _none yet_

**Phase 2 gate result + checkpoint used as init (fill in after pretrain run):** _pending_

**Run name:** `asymmetric_clip_l14_336_robertaL_d4_pt{0|1}_s42` (set automatically below).


## Load Data From Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/sample_data/

Mounted at /content/drive


In [2]:
# make data dirs
!mkdir -p /content/data/
!mkdir -p /content/data/answers
!mkdir -p /content/data/images
!mkdir -p /content/data/questions

#copy over zips (https://drive.google.com/drive/folders/1VJ1xNxo_dAGJ4ZcpaFQBo-wpdIChZkpx?usp=sharing) from drive into here
!cp -r /content/drive/MyDrive/VQA/ /content/data/zip/

In [3]:
# extract annotations (labels)
!unzip -q /content/data/zip/v2_Annotations_Train_mscoco.zip -d /content/data/answers/
!unzip -q /content/data/zip/v2_Annotations_Val_mscoco.zip -d /content/data/answers/

!rm -rf /content/data/zip/v2_Annotations_Train_mscoco.zip
!rm -rf /content/data/zip/v2_Annotations_Val_mscoco.zip

In [4]:
# extract testing data
!unzip -q /content/data/zip/v2_Questions_Test_mscoco.zip -d /content/data/questions/
!unzip -q /content/data/zip/v2_Questions_Train_mscoco.zip -d /content/data/questions/
!unzip -q /content/data/zip/v2_Questions_Val_mscoco.zip -d /content/data/questions/

!rm -rf /content/data/zip/v2_Questions_Test_mscoco.zip
!rm -rf /content/data/zip/v2_Questions_Train_mscoco.zip
!rm -rf /content/data/zip/v2_Questions_Val_mscoco.zip

In [5]:
!cp /content/drive/MyDrive/VQA_cache/vqa_images_336.h5 /content/data/

## Install dependencies

In [6]:
!pip install -q torch torchvision transformers open_clip_torch matplotlib tqdm Pillow h5py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00


## Materialize `models.py` (shared with `04_pretrain_coco_itc_itm.ipynb`)

Both notebooks write byte-identical `models.py` into the notebook's working directory.

In [7]:
%%writefile models.py
# models.py - shared across 04_pretrain_coco_itc_itm.ipynb and 04_train_clip_upgrade_v2.ipynb.
#
# Both notebooks materialize this exact content via a %%writefile cell so the
# definitions stay in sync. Edit ONE canonical source (this file in the repo
# scratchpad / both notebook cells) and the materialization is reproducible.
#
# The asymmetric cross-modal fusion is the research artifact: each block runs
# image-queries-attend-to-text FIRST, then text-queries-attend-to-the-image-side-
# output. That ordering is preserved across every block in the stack and across
# both notebooks.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip
from transformers import RobertaModel


class CLIPImageEncoder(nn.Module):
    """open_clip ViT-{B-16, L-14} @ configurable image_size, LAION-2B pretrained.
    Returns (B, N_tokens, embed_dim) where N_tokens = (image_size/patch)**2 + 1.

    Manual forward (not visual.output_tokens=True) because that flag drops the
    CLS token; we keep CLS + patches so the fusion sees the full sequence.
    """

    def __init__(self, embed_dim, image_size, model_name, pretrained,
                 freeze=False, grad_checkpointing=False):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(
            model_name, pretrained=pretrained, force_image_size=image_size,
        )
        self.visual = clip_model.visual
        del clip_model
        self.visual.proj = None

        if grad_checkpointing:
            self.visual.set_grad_checkpointing(True)

        if freeze:
            for p in self.visual.parameters():
                p.requires_grad = False

        # Read width dynamically so this works for both ViT-B (768) and ViT-L (1024).
        clip_width = self.visual.transformer.width
        self.projection = nn.Linear(clip_width, embed_dim)

        patch = self.visual.patch_size if isinstance(self.visual.patch_size, int) else self.visual.patch_size[0]
        expected_tokens = (image_size // patch) ** 2 + 1
        actual_tokens = self.visual.positional_embedding.shape[0]
        assert actual_tokens == expected_tokens, (
            f"position embedding interpolation failed: got {actual_tokens} tokens, "
            f"want {expected_tokens} (image_size={image_size}, patch={patch})"
        )
        print(f"[CLIPImageEncoder] {model_name} / {pretrained} @ {image_size} -> "
              f"{actual_tokens} tokens, width={clip_width}, grad_ckpt={grad_checkpointing}")

    def forward(self, images):
        v = self.visual
        x = v.conv1(images)
        x = x.reshape(x.shape[0], x.shape[1], -1)
        x = x.permute(0, 2, 1)

        cls = v.class_embedding.to(x.dtype).view(1, 1, -1).expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + v.positional_embedding.to(x.dtype)
        x = v.patch_dropout(x)
        x = v.ln_pre(x)

        batch_first = getattr(v.transformer, "batch_first", True)
        if not batch_first:
            x = x.permute(1, 0, 2)
        x = v.transformer(x)
        if not batch_first:
            x = x.permute(1, 0, 2)

        x = v.ln_post(x)
        return self.projection(x)


class TextEncoder(nn.Module):
    """RoBERTa text encoder. Reads hidden dim from config (works for both
    roberta-base @ 768 and roberta-large @ 1024). Bypasses the freshly-initialized
    pooler.dense. Calls gradient_checkpointing_enable() if requested.
    """
    POOLING = "last_hidden_state"

    def __init__(self, embed_dim, model_name="roberta-large",
                 freeze=False, grad_checkpointing=False):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        hidden = self.roberta.config.hidden_size
        self.projection = nn.Linear(hidden, embed_dim)

        if grad_checkpointing:
            self.roberta.gradient_checkpointing_enable()
        if freeze:
            for p in self.roberta.parameters():
                p.requires_grad = False

        assert hasattr(self.roberta, "pooler"), \
            "expected RoBERTa to expose a pooler (which we intentionally bypass)"
        print(f"[TextEncoder] {model_name} pooling={self.POOLING} "
              f"hidden_dim={hidden} grad_ckpt={grad_checkpointing}")

    def forward(self, input_ids, attention_mask):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return self.projection(out.last_hidden_state)


class CrossAttentionBlock(nn.Module):
    """Pre-norm cross-attention block: queries from one modality attend to KVs
    from another, then a residual FFN."""

    def __init__(self, embed_dim, num_heads, dropout=0.1, attn_dropout=None):
        super().__init__()
        if attn_dropout is None:
            attn_dropout = dropout
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=attn_dropout, batch_first=True)
        self.norm_ff = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, query, key_value, key_padding_mask=None):
        q = self.norm_q(query)
        kv = self.norm_kv(key_value)
        attended, attn_weights = self.cross_attn(
            q, kv, kv, key_padding_mask=key_padding_mask,
            need_weights=True, average_attn_weights=True)
        query = query + attended
        query = query + self.ff(self.norm_ff(query))
        return query, attn_weights


class AsymmetricCrossModalFusion(nn.Module):
    """Stack of `depth` asymmetric fusion blocks. Each block:
        img_attended = cross_attn_img_to_txt(query=img, kv=txt)
        txt_attended = cross_attn_txt_to_img(query=txt, kv=img_attended)
    Each block has its own learned weights -- no weight sharing across blocks.
    The asymmetric image-first ordering is the research contribution; do not flip.
    """

    def __init__(self, embed_dim, num_heads, depth=1, dropout=0.1, attn_dropout=None):
        super().__init__()
        self.depth = depth
        self.blocks_img_to_txt = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.blocks_txt_to_img = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])

    def forward(self, image_features, text_features, text_padding_mask=None):
        img, txt = image_features, text_features
        last_attn_i2t = last_attn_t2i = None
        for blk_i2t, blk_t2i in zip(self.blocks_img_to_txt, self.blocks_txt_to_img):
            img_attended, last_attn_i2t = blk_i2t(
                query=img, key_value=txt,
                key_padding_mask=text_padding_mask)
            txt_attended, last_attn_t2i = blk_t2i(
                query=txt, key_value=img_attended)
            img, txt = img_attended, txt_attended
        return img, txt, last_attn_i2t, last_attn_t2i


class AttentionPool(nn.Module):
    """Learnable-query attention pool: maps a token sequence to a single vector.
    Supports key_padding_mask for variable-length text streams.
    """

    def __init__(self, embed_dim, num_heads=8):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, key_padding_mask=None):
        q = self.query.expand(x.size(0), -1, -1)
        out, _ = self.attn(q, x, x, key_padding_mask=key_padding_mask)
        return self.norm(out.squeeze(1))


class AsymmetricVQAModelE2E(nn.Module):
    """End-to-end asymmetric cross-modal model used by BOTH pretraining and finetuning.

    Always constructs encoders + fusion + attention pools.

    If `num_answers` is not None: constructs the VQA classifier.
    If `pretrain_heads=True`: constructs ITC projection MLPs + ITM head + logit_scale.

    Use `pretrain_heads=True, num_answers=None` for pretraining.
    Use `pretrain_heads=False, num_answers=3129` for finetuning.
    `load_state_dict(strict=False)` from a pretrain checkpoint tolerates the
    missing classifier keys and unmatched pretrain-head keys.
    """

    def __init__(self,
                 num_answers,
                 embed_dim=768, num_heads=12, fusion_depth=4,
                 dropout=0.3, attn_dropout=0.2, cls_dropout=0.5,
                 image_size=336,
                 clip_model_name="ViT-L-14", clip_pretrained="laion2b_s32b_b82k",
                 text_model_name="roberta-large",
                 vision_grad_checkpointing=True,
                 text_grad_checkpointing=True,
                 pretrain_heads=False,
                 pool_num_heads=8):
        super().__init__()
        self.image_encoder = CLIPImageEncoder(
            embed_dim=embed_dim, image_size=image_size,
            model_name=clip_model_name, pretrained=clip_pretrained,
            freeze=False, grad_checkpointing=vision_grad_checkpointing,
        )
        self.text_encoder = TextEncoder(
            embed_dim=embed_dim, model_name=text_model_name,
            freeze=False, grad_checkpointing=text_grad_checkpointing,
        )
        self.fusion = AsymmetricCrossModalFusion(
            embed_dim, num_heads, depth=fusion_depth,
            dropout=dropout, attn_dropout=attn_dropout,
        )
        self.pool_img = AttentionPool(embed_dim, num_heads=pool_num_heads)
        self.pool_txt = AttentionPool(embed_dim, num_heads=pool_num_heads)

        self.classifier = None
        if num_answers is not None:
            self.classifier = nn.Sequential(
                nn.Linear(embed_dim * 2, embed_dim),
                nn.ReLU(),
                nn.Dropout(cls_dropout),
                nn.Linear(embed_dim, num_answers),
            )

        self.pretrain_heads = pretrain_heads
        if pretrain_heads:
            self.itc_proj_img = nn.Sequential(
                nn.Linear(embed_dim, embed_dim), nn.GELU(),
                nn.Linear(embed_dim, 256),
            )
            self.itc_proj_txt = nn.Sequential(
                nn.Linear(embed_dim, embed_dim), nn.GELU(),
                nn.Linear(embed_dim, 256),
            )
            self.itm_head = nn.Sequential(
                nn.Linear(2 * embed_dim, embed_dim), nn.GELU(),
                nn.Linear(embed_dim, 2),
            )
            self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def encode(self, images, input_ids, attention_mask):
        """Encoders + fusion + pooling -> (img_vec, txt_vec) each (B, embed_dim).
        Shared path used by both VQA forward and pretraining forward.
        """
        img_tokens = self.image_encoder(images)
        txt_tokens = self.text_encoder(input_ids, attention_mask)
        text_pad_mask = attention_mask == 0
        img_att, txt_att, _, _ = self.fusion(img_tokens, txt_tokens, text_pad_mask)
        img_vec = self.pool_img(img_att)
        txt_vec = self.pool_txt(txt_att, key_padding_mask=text_pad_mask)
        return img_vec, txt_vec

    def forward(self, images, input_ids, attention_mask):
        """VQA forward. Surfaces fusion attention weights for downstream viz."""
        img_tokens = self.image_encoder(images)
        txt_tokens = self.text_encoder(input_ids, attention_mask)
        text_pad_mask = attention_mask == 0
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(
            img_tokens, txt_tokens, text_pad_mask)
        img_vec = self.pool_img(img_att)
        txt_vec = self.pool_txt(txt_att, key_padding_mask=text_pad_mask)
        z = torch.cat([img_vec, txt_vec], dim=-1)
        assert self.classifier is not None, "VQA forward requires num_answers != None"
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}

    def forward_pretrain(self, images, input_ids, attention_mask):
        """Returns {img_vec, txt_vec, img_emb, txt_emb, logit_scale}.
        img_emb / txt_emb are L2-normalized 256-d projections for ITC.
        """
        assert self.pretrain_heads, "forward_pretrain requires pretrain_heads=True"
        img_vec, txt_vec = self.encode(images, input_ids, attention_mask)
        img_proj = self.itc_proj_img(img_vec)
        txt_proj = self.itc_proj_txt(txt_vec)
        img_emb = F.normalize(img_proj, dim=-1)
        txt_emb = F.normalize(txt_proj, dim=-1)
        return {
            "img_vec": img_vec, "txt_vec": txt_vec,
            "img_emb": img_emb, "txt_emb": txt_emb,
            "logit_scale": self.logit_scale,
        }


Writing models.py


## Imports and device

In [8]:
import json
import math
import random
import re
import shutil
import time
import zipfile
import urllib.request
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.amp import autocast
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.transforms import RandAugment
from tqdm import tqdm
from transformers import RobertaTokenizer

# models.py is materialized in the cell above via %%writefile.
import sys
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
from models import (
    CLIPImageEncoder,
    TextEncoder,
    CrossAttentionBlock,
    AsymmetricCrossModalFusion,
    AttentionPool,
    AsymmetricVQAModelE2E,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cuda


## Configuration

In [13]:
# Architecture (must match 04_pretrain_coco_itc_itm.ipynb exactly).
CLIP_MODEL          = "ViT-L-14"
CLIP_PRETRAINED     = "laion2b_s32b_b82k"
IMAGE_SIZE          = 336
TEXT_MODEL          = "roberta-large"
EMBED_DIM           = 768
NUM_HEADS           = 12
FUSION_DEPTH        = 4
DROPOUT             = 0.3
ATTN_DROPOUT        = 0.2
CLS_DROPOUT         = 0.5
USE_GRAD_CHECKPOINT = True

# VQA data
NUM_ANSWERS         = 3129
MAX_QUESTION_LEN    = 20
MAX_SAMPLES         = None     # use the full training set

# VQA finetuning schedule
BATCH_SIZE          = 8        # per-step
GRAD_ACCUM_STEPS    = 8        # effective batch = 64
EPOCHS              = 7
LEARNING_RATE       = 3e-5     # fusion + pools + classifier
ENCODER_LR          = 3e-6     # CLIP vision + RoBERTa (base of LLRD decay)
WARMUP_FRAC         = 0.10
LLRD_DECAY          = 0.95     # per-layer encoder LR decay (top trains full, bottom near-frozen)
WEIGHT_DECAY        = 0.01
NUM_WORKERS         = 2
SEED                = 42
GRAD_CLIP_NORM      = 1.0

# EMA
USE_EMA             = True
EMA_DECAY           = 0.9999

# Smoke flag: True for Phase 0/1/3 gate runs on small subsets. Flip to False for the real 7-epoch run.
SMOKE_TEST          = True
SMOKE_TRAIN_SAMPLES = 20000    # used when SMOKE_TEST=True (Phase 1 gate)
SMOKE_VAL_SAMPLES   = 5000

# Paths
DATA_DIR            = Path("/content/data/")
CHECKPOINT_DIR      = Path("/content/results/checkpoints")
METRICS_DIR         = Path("/content/results/metrics")
FIGURES_DIR         = Path("/content/results/figures")
PREDICTIONS_DIR     = Path("/content/results/predictions")
for d in [CHECKPOINT_DIR, METRICS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Pretrain init detection
PRETRAIN_CKPT       = CHECKPOINT_DIR / "pretrain_final.pt"
GATE_MARKER         = Path("../results/PRETRAIN_GATE_FAILED.txt")
USE_PRETRAIN        = PRETRAIN_CKPT.exists() and not GATE_MARKER.exists()
RUN_NAME            = f"asymmetric_clip_l14_336_robertaL_d4_pt{int(USE_PRETRAIN)}_s{SEED}"

print(f"[Config] {RUN_NAME} | bs={BATCH_SIZE}*accum{GRAD_ACCUM_STEPS}={BATCH_SIZE*GRAD_ACCUM_STEPS} "
      f"| epochs={EPOCHS} | image={IMAGE_SIZE} | fusion_depth={FUSION_DEPTH} "
      f"| embed_dim={EMBED_DIM} | pretrain={'YES' if USE_PRETRAIN else 'NO (LAION-2B init)'} "
      f"| smoke={SMOKE_TEST}")


[Config] asymmetric_clip_l14_336_robertaL_d4_pt0_s42 | bs=8*accum8=64 | epochs=7 | image=336 | fusion_depth=4 | embed_dim=768 | pretrain=NO (LAION-2B init) | smoke=True


In [10]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


## Download COCO captions (idempotent)

Mirror of the cell in `04_pretrain_coco_itc_itm.ipynb` so this notebook is self-sufficient. Short-circuits if the files are already present.

In [11]:
# Idempotent: download COCO 2014 captions if missing.
CAPTIONS_DIR = Path("../data/captions")
CAPTIONS_DIR.mkdir(parents=True, exist_ok=True)
CAP_TRAIN = CAPTIONS_DIR / "captions_train2014.json"
CAP_VAL   = CAPTIONS_DIR / "captions_val2014.json"

if CAP_TRAIN.exists() and CAP_VAL.exists():
    print(f"[Captions] already present at {CAPTIONS_DIR}")
else:
    zip_path = CAPTIONS_DIR / "annotations_trainval2014.zip"
    url = "http://images.cocodataset.org/annotations/annotations_trainval2014.zip"
    if not zip_path.exists():
        print(f"[Captions] downloading {url} ...")
        urllib.request.urlretrieve(url, zip_path)
        print(f"[Captions] downloaded -> {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)")
    print("[Captions] extracting captions JSONs ...")
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith("captions_train2014.json"):
                with zf.open(name) as src, open(CAP_TRAIN, "wb") as dst:
                    shutil.copyfileobj(src, dst)
            elif name.endswith("captions_val2014.json"):
                with zf.open(name) as src, open(CAP_VAL, "wb") as dst:
                    shutil.copyfileobj(src, dst)
    zip_path.unlink(missing_ok=True)
    print(f"[Captions] extracted to {CAPTIONS_DIR}")

assert CAP_TRAIN.exists() and CAP_VAL.exists(), "captions extraction failed"


[Captions] downloading http://images.cocodataset.org/annotations/annotations_trainval2014.zip ...
[Captions] downloaded -> ../data/captions/annotations_trainval2014.zip (253 MB)
[Captions] extracting captions JSONs ...
[Captions] extracted to ../data/captions


## Build the 336-px H5 image cache (idempotent)

In [14]:
# Idempotent: build vqa_images_336.h5 if missing.
# Same image set as vqa_images_384.h5, just resized to 336x336 (matches ViT-L/14 patch grid 24).
LOCAL_H5_336 = DATA_DIR / "vqa_images_336.h5"
IMAGE_SIZE_H5 = 336

if LOCAL_H5_336.exists():
    print(f"[H5] using existing cache: {LOCAL_H5_336} ({LOCAL_H5_336.stat().st_size / 1e9:.1f} GB)")
else:
    preprocess = transforms.Compose([
        transforms.Resize(IMAGE_SIZE_H5, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMAGE_SIZE_H5),
    ])
    image_dirs = [DATA_DIR / "images" / "train2014", DATA_DIR / "images" / "val2014"]
    all_paths = []
    for d in image_dirs:
        if d.exists():
            all_paths.extend(sorted(d.glob("*.jpg")))
    n = len(all_paths)
    est_gb = (n * IMAGE_SIZE_H5 * IMAGE_SIZE_H5 * 3) / 1e9
    print(f"[H5] building from {n:,} images -> ~{est_gb:.1f} GB at {LOCAL_H5_336}")

    t0 = time.time()
    with h5py.File(LOCAL_H5_336, "w") as h5f:
        imgs_ds = h5f.create_dataset(
            "images", shape=(n, IMAGE_SIZE_H5, IMAGE_SIZE_H5, 3),
            dtype=np.uint8, chunks=(1, IMAGE_SIZE_H5, IMAGE_SIZE_H5, 3),
        )
        ids_ds = h5f.create_dataset("image_ids", shape=(n,), dtype=np.int64)
        for i, p in enumerate(tqdm(all_paths, desc=f"H5@{IMAGE_SIZE_H5}")):
            image_id = int(p.stem.split("_")[-1])
            img = Image.open(p).convert("RGB")
            img = preprocess(img)
            imgs_ds[i] = np.array(img)
            ids_ds[i] = image_id
    print(f"[H5] build elapsed: {time.time() - t0:.0f}s")


[H5] using existing cache: /content/data/vqa_images_336.h5 (41.8 GB)


## Helpers: answer normalization, transforms (no horizontal flip)

In [15]:
# Answer normalization (byte-identical to 03_*), transforms (no HorizontalFlip), CLIP norm.
_ARTICLES = {"a", "an", "the"}
_PUNCT_RE = re.compile(r"[^\w\s\']")
_CONTRACTIONS = {
    "dont": "don't", "doesnt": "doesn't", "didnt": "didn't",
    "isnt": "isn't", "arent": "aren't",
    "wasnt": "wasn't", "werent": "weren't",
    "wont": "won't", "cant": "can't", "couldnt": "couldn't",
    "wouldnt": "wouldn't", "shouldnt": "shouldn't",
    "havent": "haven't", "hasnt": "hasn't", "hadnt": "hadn't",
    "thats": "that's", "whats": "what's", "wheres": "where's",
    "theres": "there's", "heres": "here's", "youre": "you're",
    "theyre": "they're", "weve": "we've", "youve": "you've",
}


def normalize_answer(s: str) -> str:
    s = s.lower().strip()
    s = _PUNCT_RE.sub(" ", s)
    s = " ".join(t for t in s.split() if t not in _ARTICLES)
    return _CONTRACTIONS.get(s, s)


def build_answer_vocab(annotations_file, top_k=3129):
    with open(annotations_file) as f:
        annotations = json.load(f)["annotations"]
    counter = Counter()
    for ann in annotations:
        for a in ann["answers"]:
            counter[normalize_answer(a["answer"])] += 1
    most_common = [ans for ans, _ in counter.most_common(top_k)]
    answer_to_idx = {ans: idx for idx, ans in enumerate(most_common)}
    idx_to_answer = {idx: ans for ans, idx in answer_to_idx.items()}
    return answer_to_idx, idx_to_answer


ANSWER_TYPES = ("yes/no", "number", "other")
ANSWER_TYPE_TO_IDX = {t: i for i, t in enumerate(ANSWER_TYPES)}

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)


class RandAugmentNoGeom(RandAugment):
    """RandAugment without geometric ops (Translate/Shear/Rotate).
    VQA labels frequently depend on spatial position; geometric ops corrupt them.
    """
    _DROP_OPS = {"TranslateX", "TranslateY", "ShearX", "ShearY", "Rotate"}

    def _augmentation_space(self, num_bins, image_size):
        space = super()._augmentation_space(num_bins, image_size)
        return {k: v for k, v in space.items() if k not in self._DROP_OPS}


def get_train_transform():
    """VQA-safe augmentation: NO HorizontalFlip (contradicts RandAugmentNoGeom rationale),
    NO Translate/Shear/Rotate. ColorJitter + photometric RandAugment + RandomErasing.
    """
    return transforms.Compose([
        RandAugmentNoGeom(num_ops=2, magnitude=9,
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.10), ratio=(0.3, 3.3), value=0),
    ])


def get_val_transform():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ])


## VQADataset (shared tokenizer)

In [16]:
class VQADataset(Dataset):
    """VQA v2 dataset with H5 image loading and soft targets.
    Tokenizer is injected (single shared instance per notebook), unlike 03_*.
    """

    def __init__(self, questions_file, annotations_file, h5_path, tokenizer,
                 answer_to_idx=None, top_k_answers=3129,
                 max_question_len=20, transform=None, max_samples=None):
        self.h5_path = Path(h5_path)
        self.max_question_len = max_question_len
        self.transform = transform or get_val_transform()
        self.tokenizer = tokenizer
        self._h5f = None

        if answer_to_idx is None:
            self.answer_to_idx, self.idx_to_answer = build_answer_vocab(
                annotations_file, top_k_answers)
        else:
            self.answer_to_idx = answer_to_idx
            self.idx_to_answer = {v: k for k, v in answer_to_idx.items()}

        with h5py.File(self.h5_path, "r") as f:
            image_ids = f["image_ids"][:]
        self.id_to_row = {int(iid): i for i, iid in enumerate(image_ids)}

        with open(questions_file) as f:
            questions_data = json.load(f)["questions"]
        with open(annotations_file) as f:
            annotations_data = json.load(f)["annotations"]

        ann_by_qid = {ann["question_id"]: ann for ann in annotations_data}
        num_answers = len(self.answer_to_idx)

        self.samples = []
        for q in questions_data:
            ann = ann_by_qid.get(q["question_id"])
            if ann is None:
                continue
            target = torch.zeros(num_answers, dtype=torch.float)
            for a in ann["answers"]:
                ans_text = normalize_answer(a["answer"])
                idx = self.answer_to_idx.get(ans_text)
                if idx is not None:
                    target[idx] += 1.0
            if target.sum() == 0:
                continue
            target /= 10.0
            atype = ann.get("answer_type", "other")
            atype_idx = ANSWER_TYPE_TO_IDX.get(atype, ANSWER_TYPE_TO_IDX["other"])
            self.samples.append({
                "question": q["question"],
                "image_id": q["image_id"],
                "answer_target": target,
                "answer_type_idx": atype_idx,
            })
            if max_samples is not None and len(self.samples) >= max_samples:
                break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        if self._h5f is None:
            self._h5f = h5py.File(self.h5_path, "r")
        row = self.id_to_row[sample["image_id"]]
        img_array = self._h5f["images"][row]
        image = Image.fromarray(img_array)
        image = self.transform(image)

        encoding = self.tokenizer(
            sample["question"],
            max_length=self.max_question_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return (
            image,
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            sample["answer_target"],
            torch.tensor(sample["answer_type_idx"], dtype=torch.long),
        )


## Build vocab + datasets + loaders

In [17]:
# Single shared tokenizer for both train and val datasets.
tokenizer = RobertaTokenizer.from_pretrained(TEXT_MODEL)

train_ann = DATA_DIR / "answers" / "v2_mscoco_train2014_annotations.json"
answer_to_idx, idx_to_answer = build_answer_vocab(train_ann, NUM_ANSWERS)
print(f"Answer vocab: {len(answer_to_idx)} classes")

train_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_train2014_questions.json",
    annotations_file=train_ann,
    h5_path=LOCAL_H5_336,
    tokenizer=tokenizer,
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_train_transform(),
    max_samples=MAX_SAMPLES,
)
val_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_val2014_questions.json",
    annotations_file=DATA_DIR / "answers" / "v2_mscoco_val2014_annotations.json",
    h5_path=LOCAL_H5_336,
    tokenizer=tokenizer,
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_val_transform(),
    max_samples=MAX_SAMPLES,
)

if SMOKE_TEST:
    train_ds = Subset(train_ds, list(range(min(SMOKE_TRAIN_SAMPLES, len(train_ds)))))
    val_ds   = Subset(val_ds,   list(range(min(SMOKE_VAL_SAMPLES,   len(val_ds)))))
    print(f"[SMOKE] train={len(train_ds)} val={len(val_ds)}")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0, prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0, prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
print(f"Train: {len(train_ds):,} samples ({len(train_loader)} batches @ batch={BATCH_SIZE})")
print(f"Val:   {len(val_ds):,} samples ({len(val_loader)} batches)")

with open(CHECKPOINT_DIR / "answer_vocab.json", "w") as f:
    json.dump(answer_to_idx, f)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Answer vocab: 3129 classes
[SMOKE] train=20000 val=5000
Train: 20,000 samples (2500 batches @ batch=8)
Val:   5,000 samples (625 batches)


## EMA wrapper (decay 0.9999)

In [18]:
# EMA wrapper — shadow lives on CPU to save ~2.6 GB of GPU memory.
# Update copies the live tensors to CPU; swap context backs up live params to CPU
# and brings the shadow onto GPU just for eval/checkpoint saving.
class ModelEMA:
    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.steps = 0
        self.shadow = {
            n: p.detach().to("cpu", copy=True)
            for n, p in model.state_dict().items()
            if torch.is_floating_point(p)
        }

    @torch.no_grad()
    def update(self, model):
        d = self.decay
        msd = model.state_dict()
        for n, shadow_p in self.shadow.items():
            live = msd[n].detach().to(shadow_p.device, dtype=shadow_p.dtype, non_blocking=True)
            shadow_p.mul_(d).add_(live, alpha=1.0 - d)
        self.steps += 1

    @torch.no_grad()
    def copy_to(self, model):
        msd = model.state_dict()
        for n, shadow_p in self.shadow.items():
            msd[n].copy_(shadow_p.to(msd[n].device, dtype=msd[n].dtype, non_blocking=True))

    def state_dict(self):
        return {"decay": self.decay, "steps": self.steps, "shadow": self.shadow}

    def load_state_dict(self, sd):
        self.decay = sd["decay"]; self.steps = sd["steps"]
        # Tolerate checkpoints saved with a GPU-resident shadow.
        self.shadow = {n: t.detach().to("cpu") for n, t in sd["shadow"].items()}


class _SwapEMA:
    """Temporarily swap live model params with EMA shadow. Backup held on CPU
    so we don't keep a duplicate set of weights on the GPU during eval.
    """
    def __init__(self, model, ema):
        self.model = model; self.ema = ema; self._backup = {}
    def __enter__(self):
        msd = self.model.state_dict()
        for n in self.ema.shadow:
            self._backup[n] = msd[n].detach().to("cpu", copy=True)
        self.ema.copy_to(self.model)
        return self.model
    def __exit__(self, exc_type, exc, tb):
        msd = self.model.state_dict()
        for n, v in self._backup.items():
            msd[n].copy_(v.to(msd[n].device, dtype=msd[n].dtype, non_blocking=True))
        self._backup.clear()


## Build model

In [19]:
model = AsymmetricVQAModelE2E(
    num_answers=NUM_ANSWERS,
    embed_dim=EMBED_DIM, num_heads=NUM_HEADS, fusion_depth=FUSION_DEPTH,
    dropout=DROPOUT, attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT,
    image_size=IMAGE_SIZE,
    clip_model_name=CLIP_MODEL, clip_pretrained=CLIP_PRETRAINED,
    text_model_name=TEXT_MODEL,
    vision_grad_checkpointing=USE_GRAD_CHECKPOINT,
    text_grad_checkpointing=USE_GRAD_CHECKPOINT,
    pretrain_heads=False,
).to(device)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"[Model] trainable={n_train:,} / total={n_total:,}")

# Sanity asserts on a dummy forward
with torch.no_grad():
    _img  = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    _ids  = torch.zeros(2, MAX_QUESTION_LEN, dtype=torch.long, device=device)
    _mask = torch.ones(2, MAX_QUESTION_LEN, dtype=torch.long, device=device)
    with autocast("cuda", dtype=torch.bfloat16):
        _logits, _ = model(_img, _ids, _mask)
    expected_tokens = (IMAGE_SIZE // 14) ** 2 + 1
    _vt = model.image_encoder(_img)
    _tt = model.text_encoder(_ids, _mask)
    assert _vt.shape == (2, expected_tokens, EMBED_DIM), f"vision={_vt.shape}"
    assert _tt.shape == (2, MAX_QUESTION_LEN, EMBED_DIM), f"text={_tt.shape}"
    assert _logits.shape == (2, NUM_ANSWERS), f"logits={_logits.shape}"
    print(f"[Sanity] vision={tuple(_vt.shape)} text={tuple(_tt.shape)} logits={tuple(_logits.shape)}")
del _img, _ids, _mask, _logits, _vt, _tt


open_clip_pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

[CLIPImageEncoder] ViT-L-14 / laion2b_s32b_b82k @ 336 -> 577 tokens, width=1024, grad_ckpt=True


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[TextEncoder] roberta-large pooling=last_hidden_state hidden_dim=1024 grad_ckpt=True
[Model] trainable=725,472,825 / total=725,472,825
[Sanity] vision=(2, 577, 768) text=(2, 20, 768) logits=(2, 3129)


## Load pretrained checkpoint (if available + gate passed)

In [20]:
if USE_PRETRAIN:
    state = torch.load(PRETRAIN_CKPT, map_location="cpu", weights_only=False)["model"]
    missing, unexpected = model.load_state_dict(state, strict=False)
    expected_loaded_prefixes = ("image_encoder.", "text_encoder.", "fusion.", "pool_img.", "pool_txt.")
    model_keys = {k for k in model.state_dict() if k.startswith(expected_loaded_prefixes)}
    missing_set = set(missing)
    actually_loaded = model_keys - missing_set
    print(f"[Load] matched {len(actually_loaded)} of {len(model_keys)} expected pretrained keys")
    not_loaded = model_keys - actually_loaded
    assert not not_loaded, f"missing pretrained tensors: {sorted(not_loaded)[:5]}..."
    # The classifier should be in 'missing' (fresh) and pretrain heads should be in 'unexpected'.
    print(f"[Load] missing (expected: classifier.*):     {[m for m in missing if 'classifier' in m][:3]}...")
    print(f"[Load] unexpected (expected: itc/itm/logit): {sorted(unexpected)[:5]}...")
else:
    print(f"[Load] No pretrained checkpoint used (pretrain={PRETRAIN_CKPT.exists()}, "
          f"gate_failed={GATE_MARKER.exists()}). Starting from LAION-2B encoder init.")


[Load] No pretrained checkpoint used (pretrain=False, gate_failed=False). Starting from LAION-2B encoder init.


## Optimizer (LLRD) + scheduler + EMA + loss

In [21]:
# Layer-wise LR decay (LLRD) on both encoders. Each transformer layer gets
# its own param group with LR = ENCODER_LR * (LLRD_DECAY ** (n_layers - layer_idx)).
NO_DECAY_KEYS = ("bias", "LayerNorm.weight", "layer_norm.weight", "ln_")

def _wd_for(name):
    return 0.0 if any(k in name for k in NO_DECAY_KEYS) else WEIGHT_DECAY

# --- RoBERTa: layers are roberta.encoder.layer.<i>.* ; embeddings are roberta.embeddings.* ---
roberta_layers = model.text_encoder.roberta.encoder.layer
n_text_layers  = len(roberta_layers)  # 24 for roberta-large

# --- CLIP visual transformer: resblocks are visual.transformer.resblocks.<i>.* ---
clip_blocks = model.image_encoder.visual.transformer.resblocks
n_vis_layers = len(clip_blocks)       # 24 for ViT-L/14

param_groups = []
seen = set()

def _add_group(named, lr):
    decay, nodecay = [], []
    for n, p in named:
        if not p.requires_grad or id(p) in seen:
            continue
        seen.add(id(p))
        (nodecay if any(k in n for k in NO_DECAY_KEYS) else decay).append(p)
    if decay:   param_groups.append({"params": decay,   "lr": lr, "weight_decay": WEIGHT_DECAY})
    if nodecay: param_groups.append({"params": nodecay, "lr": lr, "weight_decay": 0.0})

# Per-layer LLRD groups: top layer (idx n-1) gets full ENCODER_LR; layer i gets
# ENCODER_LR * LLRD_DECAY ** (n - 1 - i). Embeddings / non-transformer params get
# the bottom-most LR.
for i, layer in enumerate(roberta_layers):
    lr_i = ENCODER_LR * (LLRD_DECAY ** (n_text_layers - 1 - i))
    _add_group(layer.named_parameters(prefix=f"text_encoder.roberta.encoder.layer.{i}"), lr_i)

for i, blk in enumerate(clip_blocks):
    lr_i = ENCODER_LR * (LLRD_DECAY ** (n_vis_layers - 1 - i))
    _add_group(blk.named_parameters(prefix=f"image_encoder.visual.transformer.resblocks.{i}"), lr_i)

# Encoder bottom: embeddings + remaining encoder params not yet seen. Bottom LR.
bottom_lr = ENCODER_LR * (LLRD_DECAY ** max(n_text_layers, n_vis_layers))
_add_group([(n, p) for n, p in model.text_encoder.named_parameters(prefix="text_encoder")
            if "encoder.layer" not in n], bottom_lr)
_add_group([(n, p) for n, p in model.image_encoder.named_parameters(prefix="image_encoder")
            if "transformer.resblocks" not in n], bottom_lr)

# Everything else (fusion, pools, classifier) -> LEARNING_RATE.
_add_group([(n, p) for n, p in model.named_parameters()
            if not (n.startswith("text_encoder") or n.startswith("image_encoder"))],
           LEARNING_RATE)

optimizer = torch.optim.AdamW(param_groups)
total_steps  = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = max(1, int(round(total_steps * WARMUP_FRAC)))

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
ema = ModelEMA(model, decay=EMA_DECAY) if USE_EMA else None
criterion = nn.BCEWithLogitsLoss()

print(f"[Opt] {len(param_groups)} param groups | total steps={total_steps} warmup={warmup_steps} "
      f"({WARMUP_FRAC*100:.0f}%) | LLRD={LLRD_DECAY} | EMA decay={EMA_DECAY}")


[Opt] 102 param groups | total steps=2191 warmup=219 (10%) | LLRD=0.95 | EMA decay=0.9999


## Train + eval functions (bf16, no GradScaler)

In [22]:
def train_one_epoch_bf16(model, loader, criterion, optimizer, scheduler, ema,
                         accum_steps, grad_clip_norm, log_peak_mem=False):
    """bf16 autocast, no GradScaler. Otherwise mirrors 03_* train_one_epoch_amp."""
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    correct = 0
    total = 0
    micro_in_batch = 0
    peak_logged = not log_peak_mem

    pbar = tqdm(loader, desc="  train", leave=False)
    for step, batch in enumerate(pbar):
        images, ids, masks, answers, _atype = batch
        images  = images.to(device, non_blocking=True)
        ids     = ids.to(device, non_blocking=True)
        masks   = masks.to(device, non_blocking=True)
        answers = answers.to(device, non_blocking=True)

        with autocast("cuda", dtype=torch.bfloat16):
            logits, _ = model(images, ids, masks)
            loss = criterion(logits, answers)
            loss_scaled = loss / accum_steps

        loss_scaled.backward()

        micro_in_batch += 1
        is_step = (micro_in_batch == accum_steps) or (step == len(loader) - 1)

        if is_step:
            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            if ema is not None:
                ema.update(model)

            if not peak_logged and torch.cuda.is_available():
                peak_gb = torch.cuda.max_memory_allocated() / 1e9
                print(f"[Mem] peak after first optimizer step: {peak_gb:.2f} GB "
                      f"(bs={BATCH_SIZE}, accum={accum_steps}, ckpt={USE_GRAD_CHECKPOINT})")
                peak_logged = True

            micro_in_batch = 0

        total_loss += loss.item() * answers.size(0)
        correct += (logits.argmax(dim=1) == answers.argmax(dim=1)).sum().item()
        total += answers.size(0)

    return {"train_loss": total_loss / total, "train_acc": correct / total * 100}


@torch.no_grad()
def evaluate_with_types(model, loader, criterion, ema=None):
    swap = _SwapEMA(model, ema) if ema is not None else nullcontext()
    with swap:
        model.eval()
        total_loss = 0.0
        vqa_acc_sum = 0.0
        total = 0
        per_type_sum = np.zeros(len(ANSWER_TYPES), dtype=np.float64)
        per_type_cnt = np.zeros(len(ANSWER_TYPES), dtype=np.int64)

        for images, ids, masks, answers, atypes in tqdm(loader, desc="  eval", leave=False):
            images  = images.to(device, non_blocking=True)
            ids     = ids.to(device, non_blocking=True)
            masks   = masks.to(device, non_blocking=True)
            answers = answers.to(device, non_blocking=True)
            atypes_np = atypes.numpy()

            with autocast("cuda", dtype=torch.bfloat16):
                logits, _ = model(images, ids, masks)
                loss = criterion(logits, answers)

            total_loss += loss.item() * answers.size(0)
            preds = logits.argmax(dim=1)
            pred_soft = answers[torch.arange(answers.size(0), device=device), preds]
            vqa_scores = torch.clamp(pred_soft * 10.0 / 3.0, max=1.0).detach().cpu().numpy()

            vqa_acc_sum += float(vqa_scores.sum())
            total += answers.size(0)
            for t_idx in range(len(ANSWER_TYPES)):
                m = atypes_np == t_idx
                if m.any():
                    per_type_sum[t_idx] += float(vqa_scores[m].sum())
                    per_type_cnt[t_idx] += int(m.sum())

    out = {"val_loss": total_loss / max(1, total),
           "val_vqa_acc": vqa_acc_sum / max(1, total) * 100}
    for t_idx, t_name in enumerate(ANSWER_TYPES):
        suffix = t_name.replace("/", "")
        out[f"val_vqa_acc_{suffix}"] = (
            per_type_sum[t_idx] / per_type_cnt[t_idx] * 100 if per_type_cnt[t_idx] else float("nan")
        )
    return out


## Auto-resume detection

In [23]:
# Defensive auto-resume: regex restricted to this RUN_NAME + tight _epoch\d+$ anchor.
_epoch_re = re.compile(r"_epoch(\d+)$")
def _epoch_num(p):
    m = _epoch_re.search(p.stem)
    return int(m.group(1)) if m else None

ckpts = sorted(
    (p for p in CHECKPOINT_DIR.glob(f"{RUN_NAME}_epoch*.pt") if _epoch_num(p) is not None),
    key=_epoch_num,
)
resume_path = ckpts[-1] if ckpts else None
print(f"Resume from: {resume_path}")


Resume from: None


## Run training

Set `SMOKE_TEST = False` in the config cell before kicking off the real 7-epoch run. The smoke run uses the gate criteria from the plan (Phase 0 / Phase 1 / Phase 3).

In [24]:
def run_training(resume_from=None):
    history = []
    best_val_acc = 0.0
    best_acc_epoch = None
    start_epoch = 1

    if resume_from is not None and Path(resume_from).exists():
        ckpt = torch.load(resume_from, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if ckpt.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        if ema is not None and ckpt.get("ema_state_dict") is not None:
            ema.load_state_dict(ckpt["ema_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        history_file = METRICS_DIR / f"{RUN_NAME}_history.json"
        if history_file.exists():
            with open(history_file) as f:
                history = json.load(f)
        if history:
            best_entry = max(history, key=lambda h: h.get("val_vqa_acc", 0))
            best_val_acc = best_entry.get("val_vqa_acc", 0)
            best_acc_epoch = best_entry["epoch"]
        print(f"Resumed from epoch {ckpt['epoch']}; best so far {best_val_acc:.2f}% @ epoch {best_acc_epoch}")
        del ckpt
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    for epoch in range(start_epoch, EPOCHS + 1):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.empty_cache()
        t0 = time.time()
        train_m = train_one_epoch_bf16(
            model, train_loader, criterion, optimizer, scheduler, ema,
            accum_steps=GRAD_ACCUM_STEPS, grad_clip_norm=GRAD_CLIP_NORM,
            log_peak_mem=(epoch == start_epoch),
        )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # Only run the EMA eval. The live-weights eval was a diagnostic that
        # doubled eval time and forced an extra forward pass through the full
        # val set every epoch -- drop it to stay under the 14.5 GiB budget.
        val_m_ema  = evaluate_with_types(model, val_loader, criterion, ema=ema)
        elapsed = time.time() - t0
        peak_gb = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0

        epoch_data = {
            "epoch": epoch, **train_m,
            "val_loss":            val_m_ema["val_loss"],
            "val_vqa_acc":         val_m_ema["val_vqa_acc"],
            "val_vqa_acc_yesno":   val_m_ema["val_vqa_acc_yesno"],
            "val_vqa_acc_number":  val_m_ema["val_vqa_acc_number"],
            "val_vqa_acc_other":   val_m_ema["val_vqa_acc_other"],
            "peak_mem_gb":         round(peak_gb, 2),
            "elapsed_s":           round(elapsed, 1),
        }
        history.append(epoch_data)
        print(f"  Epoch {epoch}/{EPOCHS} | loss {train_m['train_loss']:.4f} | "
              f"train {train_m['train_acc']:.2f}% | "
              f"vqa(ema) {val_m_ema['val_vqa_acc']:.2f}% | "
              f"y/n {val_m_ema['val_vqa_acc_yesno']:.2f}% | "
              f"num {val_m_ema['val_vqa_acc_number']:.2f}% | "
              f"oth {val_m_ema['val_vqa_acc_other']:.2f}% | "
              f"peak {peak_gb:.1f}GB | {elapsed:.0f}s")

        live_ckpt = CHECKPOINT_DIR / f"{RUN_NAME}_epoch{epoch}.pt"
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "ema_state_dict": ema.state_dict() if ema is not None else None,
            "metrics": epoch_data,
        }, live_ckpt)

        if val_m_ema["val_vqa_acc"] > best_val_acc:
            best_val_acc = val_m_ema["val_vqa_acc"]
            best_acc_epoch = epoch
            torch.save({"model_state_dict": model.state_dict(), **epoch_data},
                       CHECKPOINT_DIR / f"{RUN_NAME}_best_live.pt")
            with (_SwapEMA(model, ema) if ema is not None else nullcontext()):
                torch.save({"model_state_dict": model.state_dict(), **epoch_data},
                           CHECKPOINT_DIR / f"{RUN_NAME}_best_ema.pt")
            print(f"    New best (EMA): {best_val_acc:.2f}%")

        keep = {live_ckpt}
        if best_acc_epoch is not None:
            keep.add(CHECKPOINT_DIR / f"{RUN_NAME}_epoch{best_acc_epoch}.pt")
        for stale in CHECKPOINT_DIR.glob(f"{RUN_NAME}_epoch*.pt"):
            if stale not in keep:
                stale.unlink(missing_ok=True)

        with open(METRICS_DIR / f"{RUN_NAME}_history.json", "w") as f:
            json.dump(history, f, indent=2)

    print(f"\nDone. Best EMA val_vqa_acc: {best_val_acc:.2f}% @ epoch {best_acc_epoch}")
    return history


history_v2 = run_training(resume_from=resume_path)


  train:   0%|          | 8/2500 [00:45<4:03:25,  5.86s/it]

[Mem] peak after first optimizer step: 14.82 GB (bs=8, accum=8, ckpt=True)


OutOfMemoryError: CUDA out of memory. Tried to allocate 164.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 149.81 MiB is free. Including non-PyTorch memory, this process has 14.41 GiB memory in use. Of the allocated memory 13.11 GiB is allocated by PyTorch, and 1.17 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Evaluate best EMA checkpoint

In [ ]:
# Load best EMA checkpoint into a fresh model and eval on the full val loader.
best_ema_path = CHECKPOINT_DIR / f"{RUN_NAME}_best_ema.pt"
assert best_ema_path.exists(), f"missing {best_ema_path}"

eval_model = AsymmetricVQAModelE2E(
    num_answers=NUM_ANSWERS,
    embed_dim=EMBED_DIM, num_heads=NUM_HEADS, fusion_depth=FUSION_DEPTH,
    dropout=DROPOUT, attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT,
    image_size=IMAGE_SIZE,
    clip_model_name=CLIP_MODEL, clip_pretrained=CLIP_PRETRAINED,
    text_model_name=TEXT_MODEL,
    vision_grad_checkpointing=False,
    text_grad_checkpointing=False,
    pretrain_heads=False,
).to(device)
state = torch.load(best_ema_path, map_location=device, weights_only=False)
eval_model.load_state_dict(state["model_state_dict"])
eval_model.eval()

metrics = evaluate_with_types(eval_model, val_loader, nn.BCEWithLogitsLoss())
print(f"\nBest EMA checkpoint: {best_ema_path.name}")
for k, v in [
    ("Overall val_vqa_acc", metrics["val_vqa_acc"]),
    ("  Yes/No",            metrics["val_vqa_acc_yesno"]),
    ("  Number",            metrics["val_vqa_acc_number"]),
    ("  Other",             metrics["val_vqa_acc_other"]),
]:
    print(f"{k:<24} {v:.2f}%")
print(f"{'val_loss':<24} {metrics['val_loss']:.4f}")


## Plots

In [ ]:
histories = {RUN_NAME: history_v2}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in histories.items():
    epochs = [h["epoch"] for h in hist]
    axes[0].plot(epochs, [h["train_loss"] for h in hist], label=f"{name} train")
    axes[0].plot(epochs, [h["val_loss"]   for h in hist], "--", label=f"{name} val")
    axes[1].plot(epochs, [h["train_acc"]            for h in hist],        label=f"{name} train acc")
    axes[1].plot(epochs, [h["val_vqa_acc"]          for h in hist], "--",  label=f"{name} val (EMA)")
    axes[1].plot(epochs, [h["val_vqa_acc_yesno"]    for h in hist], ":",   label=f"{name} val y/n")
    axes[1].plot(epochs, [h["val_vqa_acc_number"]   for h in hist], ":",   label=f"{name} val num")
    axes[1].plot(epochs, [h["val_vqa_acc_other"]    for h in hist], ":",   label=f"{name} val other")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss")
axes[1].set(xlabel="Epoch", ylabel="Accuracy (%)", title="VQA Accuracy (overall + per-type)")
for ax in axes: ax.legend(fontsize=8, loc="best")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_curves_v2.png", dpi=150, bbox_inches="tight")
plt.show()

cats = ["yes/no", "number", "other"]
vals = [metrics[f"val_vqa_acc_{c.replace('/', '')}"] for c in cats]
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(cats, vals, color="steelblue")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("VQA Accuracy (%)")
ax.set_title(f"Per-question-type val accuracy ({RUN_NAME}, best EMA)")
ax.axhline(metrics["val_vqa_acc"], color="gray", linestyle="--", alpha=0.7,
           label=f"Overall: {metrics['val_vqa_acc']:.2f}%")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "per_type_v2.png", dpi=150, bbox_inches="tight")
plt.show()


## Copy over VQA test data

In [ ]:
!mkdir -p /content/data/zip/
!mkdir -p /content/data/images/
!cp /content/drive/MyDrive/test2015.zip /content/data/zip/

In [ ]:
!unzip -q /content/data/zip/test2015.zip -d /content/data/images/

## Test-set predictions export (VQA-v2 submission format)

In [ ]:
class VQATestDataset(Dataset):
    """VQA test2015 split: questions + raw JPEGs, no annotations.
    Resize-shortest + center-crop to IMAGE_SIZE; CLIP-normalize on the fly.
    """
    def __init__(self, questions_file, images_dir, tokenizer,
                 max_question_len=MAX_QUESTION_LEN, image_size=IMAGE_SIZE, max_samples=None):
        self.images_dir = Path(images_dir) / "test2015"
        self.max_question_len = max_question_len
        self.transform = transforms.Compose([
            transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
        ])
        self.tokenizer = tokenizer
        with open(questions_file) as f:
            self.samples = json.load(f)["questions"]
        if max_samples is not None:
            self.samples = self.samples[:max_samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img_path = self.images_dir / f"COCO_test2015_{s['image_id']:012d}.jpg"
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        enc = self.tokenizer(
            s["question"], max_length=self.max_question_len,
            padding="max_length", truncation=True, return_tensors="pt",
        )
        return (image, enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0), s["question_id"])


@torch.no_grad()
def predict_test(model, loader, idx_to_answer):
    model.eval()
    predictions = []
    for images, input_ids, attention_mask, question_ids in tqdm(loader, desc="predict"):
        images         = images.to(device, non_blocking=True)
        input_ids      = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        with autocast("cuda", dtype=torch.bfloat16):
            logits, _ = model(images, input_ids, attention_mask)
        preds = logits.argmax(dim=1).cpu().tolist()
        for qid, p in zip(question_ids.tolist(), preds):
            predictions.append({"question_id": int(qid), "answer": idx_to_answer[int(p)]})
    return predictions


TEST_QUESTIONS_FILE = DATA_DIR / "questions" / "v2_OpenEnded_mscoco_test2015_questions.json"
if TEST_QUESTIONS_FILE.exists():
    test_ds = VQATestDataset(
        questions_file=TEST_QUESTIONS_FILE,
        images_dir=DATA_DIR / "images",
        tokenizer=tokenizer,
        max_question_len=MAX_QUESTION_LEN,
        image_size=IMAGE_SIZE,
        max_samples=None,
    )
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True,
                             persistent_workers=NUM_WORKERS > 0,
                             prefetch_factor=4 if NUM_WORKERS > 0 else None)
    print(f"Test: {len(test_ds):,} samples ({len(test_loader)} batches)")

    preds = predict_test(eval_model, test_loader, idx_to_answer)
    # Strip the pt suffix in the filename so the submission name matches the deliverables.
    base = RUN_NAME.rsplit("_s", 1)[0]
    out_path = PREDICTIONS_DIR / f"{base}_test_predictions.json"
    with open(out_path, "w") as f:
        json.dump(preds, f)
    print(f"Saved {len(preds):,} predictions -> {out_path}")
else:
    print(f"[Skip] test questions not found at {TEST_QUESTIONS_FILE}")
